### Finding English Versions of Manual Transcripts (not auto-generated)

In [ ]:
!pip install youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.0/485.0 kB 6.6 MB/s eta 0:00:00


In [ ]:
# list all English versions available for a video
from youtube_transcript_api import YouTubeTranscriptApi

# Initialize the API
ytt_api = YouTubeTranscriptApi()

# Replace with your video ID # yjDrpVvpJMM
video_id = "CAjkJ2kSMmE" # "f2SggVSEHcY" # "7ydqfXiDssY" # "PSMXZPjJwAw" # "vR_KQZbj9rM"  # "sy205qhJvuI"

try:
    # Get the list of all available transcripts
    transcript_list = ytt_api.list(video_id)

    # Find and print all English variants
    print("Available English transcripts:")
    for transcript in transcript_list:
        # Check if the language code starts with 'en'
        if transcript.language_code.startswith('en'):
            print(f"- Language: {transcript.language}")
            print(f"  Language Code: {transcript.language_code}")
            print(f"  Manually Created: {not transcript.is_generated}")
            print("  ---")

except Exception as e:
    print(f"An error occurred: {e}")

Available English transcripts:
- Language: English
  Language Code: en
  Manually Created: True
  ---
- Language: English (United States)
  Language Code: en-US
  Manually Created: True
  ---
- Language: English (auto-generated)
  Language Code: en
  Manually Created: False
  ---


### Transcript Extraction

In [ ]:
# function for the actual extraction of the video transcript
# which is not autogenrated but manual and is in English


def get_english_transcript(video_id):
    """
    Simple function to get English transcript for a YouTube video.
    Prefers manual transcripts, falls back to auto-generated if needed.
    """
    try:
        # Get available transcripts
        transcript_list = YouTubeTranscriptApi().list(video_id)

        # Find all English variants
        english_codes = []
        for transcript in transcript_list:
            if transcript.language_code.startswith('en'):
                english_codes.append(transcript.language_code)
                print(f"Found: {transcript.language} ({transcript.language_code}) - {'Auto' if transcript.is_generated else 'Manual'}")

        if not english_codes:
            print("No English transcripts found")
            return None

        # Try to get manual transcript first
        try:
            transcript = transcript_list.find_manually_created_transcript(english_codes)
            print(f"✓ Using MANUAL transcript: {transcript.language_code}")
        except:
            # Fallback to any English transcript
            transcript = transcript_list.find_transcript(english_codes)
            print(f"○ Using AUTO-GENERATED transcript: {transcript.language_code}")

        # Fetch and return the data
        return transcript.fetch()

    except Exception as e:
        print(f"Error: {e}")
        return None

# Simple usage
video_id = "CAjkJ2kSMmE"  # Your video ID
transcript = get_english_transcript(video_id)

if transcript:
    print(f"\nThe extracted lines:")
    for i, line in enumerate(transcript):
        print(f"{i+1}. {line.text}")

    print(f"\nTotal lines: {len(transcript)}")

    # If you need it as plain text:
    full_text = " ".join([line.text for line in transcript])
    print(f"\nThe complete transcript:")
    print(full_text + "...")
else:
    print("No transcript available")

Found: English (en) - Manual
Found: English (United States) (en-US) - Manual
Found: English (auto-generated) (en) - Auto
✓ Using MANUAL transcript: en

The extracted lines:
1. Tornadoes can
2. happen at any time
3. and can turn your life 
4. upside down.
5. Quite literally.
6. Doesn't matter
7. if you're in your car, 
8. home,
9. or even an airplane,
10. tornadoes don't care.
11. They have one thing on their mind:
12. destruction.
13. Today,
14. we'll look at some of the worst-case
15. tornado scenarios
16. and give you tips
17.  on how to survive them
18. and stay on the ground.
19. To do this,
20. I threw myself on the path 
21. of a tornado in six 
22. different locations.
23. But for legal purposes,
24. and my will to live,
25. we simulated it all
26. with the help of Tuxedo Lab’s
27. game Teardown.
28. Let’s just say...
29. it didn't go so well.
30. Every year in the United States,
31. tornadoes cause
32. about 400 million dollars
33.  in damage
34. and kill about 70 people.
35. H

### Chapter Extraction

In [5]:
# testing chapter extraction
import requests
import re
import os

def get_video_description(video_id, api_key):
    """
    Fetches the video description using the YouTube Data API.
    """
    url = f"https://www.googleapis.com/youtube/v3/videos?part=snippet&id={video_id}&key={api_key}"
    response = requests.get(url)
    data = response.json()

    try:
        description = data['items'][0]['snippet']['description']
        return description, data
    except (KeyError, IndexError):
        return None

# Your API key here
api_key = os.getenv("YT_DATA_API_KEY")
description, data = get_video_description("CAjkJ2kSMmE", api_key)

In [7]:
print(description)

Tornadoes can happen at any time and can turn your life upside down. Quite literally. Doesn't matter if you're in your car, home, or even an airplane, tornadoes don't care. They have one thing on their mind: destruction. Today, we'll look at some of the worst-case tornado scenarios and give you tips on how to survive them and stay on the ground.

00:00 Simulating The Deadliest Tornado Scenarios
04:21 SURVIVING A TORNADO IN A HOUSE
06:25 SURVIVING A TORNADO IN A SKYSCRAPER
09:09 SURVIVING A TORNADO IN A CAR
10:48 SURVIVING A TORNADO IN A PLANE



Questions or concerns? Contact us at https://underknown.com/contact/
Interested in sponsoring our episodes or collaborating? Email us: contact@underknown.com 

Produced by the makers of What If. Check out our other channels:
What If - https://bit.ly/youtube-What-If
Aperture: https://bit.ly/aperture-show

Whether it’s an earthquake, mudslide or shark attack, survive whatever awaits you. How To Survive shows how to endure life-threatening moments

In [ ]:
# function to parse the chapters out of the video description
def parse_chapters(description):
    """
    Parses the video description for timestamps (chapters).
    The pattern looks for lines starting with MM:SS or HH:MM:SS followed by text.
    """
    # Regex to capture: (Time) (Text)
    # Time pattern: (?:\\d{1,2}:)?\\d{1,2}:\\d{2} matches MM:SS or HH:MM:SS
    # Text pattern: (.+) matches everything else on the line

    # We use re.IGNORECASE for flexibility and re.MULTILINE to check line by line
    chapter_pattern = r"((?:\d{1,2}:)?\d{1,2}:\d{2})\s*(.+)"

    chapters = []

    # Use finditer to process all matches
    for match in re.finditer(chapter_pattern, description):
        timestamp = match.group(1).strip()
        title = match.group(2).strip()

        # Simple validation: YouTube chapters must start from 0:00,
        # but the general parsing is robust enough for any timestamp.

        chapters.append({
            "timestamp": timestamp,
            "title": title
        })

    return chapters

In [ ]:
# function to get the description content of the video
def get_video_description(video_id, api_key):
    """
    Fetches the video description using the YouTube Data API.
    """
    url = f"https://www.googleapis.com/youtube/v3/videos?part=snippet&id={video_id}&key={api_key}"
    response = requests.get(url)
    data = response.json()

    try:
        description = data['items'][0]['snippet']['description']
        return description
    except (KeyError, IndexError):
        return None

###--------------------Enhanced Version--------------------------

In [ ]:
def parse_chapters_conservative(description):
    """
    Very conservative chapter parser - only matches clear timestamp patterns
    """
    if not description:
        return []

    chapters = []

    # Look for very specific timestamp patterns
    patterns = [
        r'^(\d{1,2}:\d{2})\s+(.+)',  # MM:SS followed by text
        r'^(\d{1,2}:\d{2}:\d{2})\s+(.+)',  # HH:MM:SS followed by text
    ]

    lines = description.split('\n')

    for line in lines:
        for pattern in patterns:
            match = re.match(pattern, line.strip())
            if match:
                timestamp = match.group(1)
                title = match.group(2).strip()

                # Additional validation: timestamp should make sense
                parts = timestamp.split(':')
                if len(parts) == 2:
                    min, sec = int(parts[0]), int(parts[1])
                    if 0 <= sec < 60:  # Seconds should be valid
                        start_seconds = min * 60 + sec
                        chapters.append({
                            "start_seconds": start_seconds,
                            "timestamp": timestamp,
                            "title": title
                        })
                        break
                elif len(parts) == 3:
                    hour, min, sec = int(parts[0]), int(parts[1]), int(parts[2])
                    if 0 <= min < 60 and 0 <= sec < 60:
                        start_seconds = hour * 3600 + min * 60 + sec
                        chapters.append({
                            "start_seconds": start_seconds,
                            "timestamp": timestamp,
                            "title": title
                        })
                        break

    return sorted(chapters, key=lambda x: x['start_seconds'])

In [ ]:
# --- EXECUTION ---

# 1. Your API key and Video ID (replace these)
api_key = os.getenv("YT_DATA_API_KEY")
video_id = "QC__nuqOSBI"    # "UyO3KQ3F0gc" # "CAjkJ2kSMmE"

# 2. Fetch the Description
description = get_video_description(video_id, api_key)

if description:
    # 3. Parse the Chapters
    # chapters_list = parse_chapters(description)
    # Use the robust parser
    chapters_list = parse_chapters_conservative(description)

    print(f"--- Chapters Found for Video ID: {video_id} ---")
    if chapters_list:
        for chapter in chapters_list:
            print(f"[{chapter['timestamp']}] {chapter['title']}")
    else:
        print("No chapter timestamps found in the description.")
else:
    print("Failed to retrieve video description. Check API key, video ID, and network connection.")

--- Chapters Found for Video ID: QC__nuqOSBI ---
No chapter timestamps found in the description.


### More Robust Solution addressing all cases to extract video chapters

In [ ]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.3 MB/s eta 0:00:00


In [ ]:
import yt_dlp
import json

def extract_chapters_yt_dlp(video_url):
    """
    Extracts chapters and their time ranges from a YouTube video using yt-dlp,
    with explicit type casting to prevent float/int formatting errors.
    """
    ydl_opts = {
        'skip_download': True,        # Don't download the video
        'writesubtitles': False,      # Don't download subtitles/transcript
        'getchapters': True,          # Ensure chapter metadata is fetched
        'force_generic_extractor': True,
        'quiet': True,
        'extract_flat': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            # Extract info—this is fast since we skip the download
            info = ydl.extract_info(video_url, download=False)

            chapters = info.get('chapters')

            if chapters:
                formatted_chapters = []
                for chapter in chapters:
                    start_sec = chapter['start_time']

                  
                    # 1. Ensure start_sec is an integer for accurate time calculation.
                    # 2. Use divmod for robust calculation of minutes and seconds.

                    # Convert to integer, discarding milliseconds
                    total_seconds = int(start_sec)

                    # Use divmod to get minutes and remaining seconds
                    minutes, seconds = divmod(total_seconds, 60)

                    # Format as a string, ensuring two digits (e.g., 04:21)
                    timestamp = f"{minutes:02d}:{seconds:02d}"
                   

                    formatted_chapters.append({
                        "timestamp": timestamp,
                        "title": chapter['title']
                    })

                return formatted_chapters
            else:
                return []

    except Exception as e:
        # Print the exception for debugging, but be aware of the exact error
        print(f"An error occurred during extraction: {e}")
        return []

In [ ]:
# --- EXECUTION: Test Chapter Extraction ---
# Replace this with your actual video URL if different
video_url =  "https://www.youtube.com/watch?v=QC__nuqOSBI" # "https://www.youtube.com/watch?v=CAjkJ2kSMmE"

chapters_list = extract_chapters_yt_dlp(video_url)

if chapters_list:
    print("--- Extracted Chapters ---")
    for chapter in chapters_list:
        print(f"[{chapter['timestamp']}] {chapter['title']}")
else:
    # This message is more likely to mean chapters truly don't exist in the metadata
    print("No structured chapters found in the video metadata.")
    print("No structured chapters found using yt-dlp.")

--- Extracted Chapters ---
[00:00] Intro
[01:25] Seal the door
[02:07] Metal is forbidden
[02:56] Keep your distance
[03:34] Avoid superheating
[04:22] Dont rush


### Merging the transcript extraction with youtube-transcript-api and chapter extraction with yt_dlp

In [ ]:
# Comprehensive code for transcript extraction with chapters

from youtube_transcript_api import YouTubeTranscriptApi
import yt_dlp

def get_english_transcript(video_id):
    """
    Simple function to get English transcript for a YouTube video.
    Prefers manual transcripts, falls back to auto-generated if needed.
    """
    try:
        # Get available transcripts
        transcript_list = YouTubeTranscriptApi().list(video_id)

        # Find all English variants
        english_codes = []
        for transcript in transcript_list:
            if transcript.language_code.startswith('en'):
                english_codes.append(transcript.language_code)
                print(f"Found: {transcript.language} ({transcript.language_code}) - {'Auto' if transcript.is_generated else 'Manual'}")

        if not english_codes:
            print("No English transcripts found")
            return None

        # Try to get manual transcript first
        try:
            transcript = transcript_list.find_manually_created_transcript(english_codes)
            #transcript = transcript_list.find_generated_transcript(english_codes)
            print(f"✓ Using MANUAL transcript: {transcript.language_code}")
            #print(f"○ Using AUTO_GENERATED transcript: {transcript.language_code}")
        except:
            # Fallback to any English transcript
            transcript = transcript_list.find_transcript(english_codes)
            print(f"○ Just Using an English transcript: {transcript.language_code}")

        # Fetch and return the data
        return transcript.fetch()

    except Exception as e:
        print(f"Error: {e}")
        return None

def extract_chapters_yt_dlp(video_url):
    """
    Extracts chapters and their time ranges from a YouTube video using yt-dlp,
    with explicit type casting to prevent float/int formatting errors.
    """
    ydl_opts = {
        'skip_download': True,
        'writesubtitles': False,
        'getchapters': True,
        'force_generic_extractor': True,
        'quiet': True,
        'extract_flat': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=False)
            chapters = info.get('chapters')

            if chapters:
                formatted_chapters = []
                for chapter in chapters:
                    start_sec = chapter['start_time']
                    total_seconds = int(start_sec)
                    minutes, seconds = divmod(total_seconds, 60)
                    timestamp = f"{minutes:02d}:{seconds:02d}"

                    formatted_chapters.append({
                        "start_sec": total_seconds,
                        "timestamp": timestamp,
                        "title": chapter['title']
                    })

                return formatted_chapters
            else:
                return []

    except Exception as e:
        print(f"An error occurred during extraction: {e}")
        return []

def segment_transcript_by_chapters(transcript, chapters):
    """
    Segment the transcript by chapter boundaries
    """
    if not chapters:
        # No chapters found, return entire transcript as one segment
        full_text = " ".join([line.text for line in transcript])
        return [{
            "chapter_title": "Full Video",
            "start_time": 0,
            "end_time": transcript[-1].start + transcript[-1].duration,
            "text": full_text
        }]

    segmented_content = []

    # Add segments for each chapter
    for i, chapter in enumerate(chapters):
        chapter_start = chapter["start_sec"]

        # Determine chapter end time (next chapter's start or end of video)
        if i + 1 < len(chapters):
            chapter_end = chapters[i + 1]["start_sec"]
        else:
            # Last chapter goes until end of video
            chapter_end = transcript[-1].start + transcript[-1].duration

        # Get all transcript lines within this chapter
        chapter_lines = [
            line for line in transcript
            if chapter_start <= line.start < chapter_end
        ]

        # Combine text for this chapter
        chapter_text = " ".join([line.text for line in chapter_lines])

        segmented_content.append({
            "chapter_title": chapter["title"],
            "start_time": chapter_start,
            "end_time": chapter_end,
            "text": chapter_text,
            "line_count": len(chapter_lines)
        })

    return segmented_content

In [ ]:
# Main execution
video_id = "CAjkJ2kSMmE"  #  "T_JkekrO_Qk"  # "QC__nuqOSBI" # "BhzjcmK0ik8"   #  #
video_url = f"https://www.youtube.com/watch?v={video_id}"

print("🎬 Processing Video:", video_id)
print("=" * 50)

# Step 1: Get transcript
print("\n📝 Step 1: Extracting Transcript...")
transcript = get_english_transcript(video_id)

if not transcript:
    print("❌ Failed to extract transcript")
    exit()

print(f"✅ Transcript extracted: {len(transcript)} lines")

# Step 2: Get chapters
print("\n📖 Step 2: Extracting Chapters...")
chapters = extract_chapters_yt_dlp(video_url)

if chapters:
    print(f"✅ Chapters found: {len(chapters)} chapters")
    for chapter in chapters:
        print(f"   - [{chapter['timestamp']}] {chapter['title']}")
else:
    print("ℹ️  No chapters found in video")

# Step 3: Combine transcript and chapters
print("\n🔗 Step 3: Combining Transcript with Chapters...")
segmented_content = segment_transcript_by_chapters(transcript, chapters)

# Display results
print("\n" + "=" * 50)
print("🎉 FINAL SEGMENTED CONTENT")
print("=" * 50)

for i, segment in enumerate(segmented_content):
    print(f"\n--- Segment {i+1}: {segment['chapter_title']} ---")
    print(f"⏱️  Time: {segment['start_time']:.0f}s - {segment['end_time']:.0f}s")
    print(f"📄 Lines: {segment.get('line_count', 'N/A')}")

    # Show preview of text
    preview = segment['text']
    #if len(preview) > 200:
        #preview = preview[:200] + "..."
    print(f"📝 Preview: {preview}")

# Optional: Save to file
print(f"\n💾 Summary: {len(segmented_content)} segments created")
if len(segmented_content) == 1:
    print("ℹ️  Video has no chapters - using single segment")
else:
    print("✅ Video successfully segmented by chapters!")

🎬 Processing Video: CAjkJ2kSMmE

📝 Step 1: Extracting Transcript...
Found: English (en) - Manual
Found: English (United States) (en-US) - Manual
Found: English (auto-generated) (en) - Auto
✓ Using MANUAL transcript: en
✅ Transcript extracted: 628 lines

📖 Step 2: Extracting Chapters...
✅ Chapters found: 5 chapters
   - [00:00] Simulating The Deadliest Tornado Scenarios
   - [04:21] SURVIVING A TORNADO IN A HOUSE
   - [06:25] SURVIVING A TORNADO IN A SKYSCRAPER
   - [09:09] SURVIVING A TORNADO IN A CAR
   - [10:48] SURVIVING A TORNADO IN A PLANE

🔗 Step 3: Combining Transcript with Chapters...

🎉 FINAL SEGMENTED CONTENT

--- Segment 1: Simulating The Deadliest Tornado Scenarios ---
⏱️  Time: 0s - 261s
📄 Lines: 169
📝 Preview: Tornadoes can happen at any time and can turn your life  upside down. Quite literally. Doesn't matter if you're in your car,  home, or even an airplane, tornadoes don't care. They have one thing on their mind: destruction. Today, we'll look at some of the worst-case